In [0]:
spark

In [0]:
from pyspark.sql import SparkSession
spark1 = SparkSession.builder.appName('Batch28').getOrCreate()


In [0]:
spark1

In [0]:
%sql
show catalogs

In [0]:
%sql
use catalog batch28_catalog

In [0]:
%sql
show schemas

In [0]:
%sql
use schema test_b28_schema

In [0]:
%sql
CREATE TABLE employees (
  employee_id INT,
  first_name STRING,
  last_name STRING,
  email STRING,
  phone STRING,
  hire_date DATE,
  department STRING
);

In [0]:
%sql
show volumes

In [0]:
dbutils.fs.ls('batch28_catalog.test_b28_schema.bronze')

In [0]:
path =r'/Volumes/batch28_catalog/test_b28_schema/bronze/customers-2000000.csv'

df =spark.read.csv(path,header=True,inferSchema=True)
df.display()
df.printSchema()

In [0]:
# print column names from dataframe
df.columns

# print count of columns in dataframe 
print(f'count of column is:{len(df.columns)}')
# print count of records in dataframe
df.count()

In [0]:
df.dtypes

In [0]:
# create list of columns have integer type and tsring type from dtaframe
int_col=[]
str_col=[]
for col_info in df.dtypes:
    print(col_info)
    if col_info[1] == 'int':
        int_col.append(col_info[0])
    elif col_info[1] == 'string':
        str_col.append(col_info[0])
    else:
        print(f'column {col_info[0]} is not int or string')
print(f'integer columns are:{int_col}')
print(f'string columns are:{str_col}')



In [0]:
df.display()

In [0]:
# select specific columns from dataframe:
df1 = df.select('Index','First Name','Last Name')
df1.display()

In [0]:
# rename columns from dataframe
# withColumnRenamed - this is used to rename column from dataframe
# syntax :- withColumnRenamed(existing_col_name,new_col_name)
from pyspark.sql.functions import *
df2 = df
for col_name in df.columns:
    df2=df2.withColumnRenamed(col_name,(col_name.replace(" ","_")).lower())
df2.display()


In [0]:
# add column in dataframe using withColumn
# lit :- this method is used to add constant value in dataframe
df3=df2.withColumn("owner",lit("Batch28"))
df3.display()

In [0]:
# add todays date column in dataframe
df4=df3.withColumn("process_run_date",current_date())\
       .withColumn("process_run_time",current_timestamp())\
       .withColumn("subscription_year",year(col('subscription_date')))\
        .withColumn("subscription_month",month(col('subscription_date')))\
        .withColumn("subscription_date",dayofmonth(col('subscription_date')))
        

df4.display()

In [0]:
df5 =df.withColumn("today_date",current_date())\
        .withColumn("subscription_age",months_between(col('today_date'),col('Subscription Date')))
df5.display() 


In [0]:
# read parquet file 
file_path =r'/Volumes/batch28_catalog/test_b28_schema/bronze/weather.parquet'
parquet_df = spark.read.parquet(file_path)
parquet_df.display()
# parquet_df.printSchema()

In [0]:
# read csv file with header and enforce the schema
# schema enforcement..

from pyspark.sql.types import *
path =r'/Volumes/batch28_catalog/test_b28_schema/bronze/batch18_student_without_header.csv'
schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("dept", IntegerType(), True)
])
df=spark.read.csv(path,schema = schema)
df.display()

In [0]:
# add new column with reference to exitsting column
# drop :- used to delete the column from dataframe
from pyspark.sql.functions import *
df1=df.withColumn('updated_age',col('age')+10)\
     . drop (col('age'))\
     .withColumnRenamed('updated_age','age')
df1.display()

In [0]:
target_path =r'/Volumes/batch28_catalog/test_b28_schema/silver/employee'
df1.write.mode('append').csv(target_path)

In [0]:
path=r'/Volumes/batch28_catalog/test_b28_schema/bronze/customers-2000000.csv'
df=spark.read.csv(path,header=True,inferSchema=True)
df.display()


In [0]:
# coalesce - used to reduce number of partitions
# mode :- overwrite - remove existing file and create new
        # append - craete new file and add data
# partition by  :- create folders by specific column values
# header - true - to populate column names in file
# write.csv - used to write data in file
target_path =r'/Volumes/batch28_catalog/test_b28_schema/silver/customers'
df.coalesce(1).write.mode('overwrite').partitionBy("Subscription Date").option('header','true').csv(target_path)

In [0]:
#hey I successfully commited